This is the second version of stackoverflow.db, where I specified that some fields need to be INTEGER type. However votes table is not finalised, I need to recreate it.

I am going to expore and check tables in stackoverflow.db

In [1]:
import sqlite3
import pandas as pd
import os

Check database size

In [2]:
db_size = os.path.getsize("stackoverflow.db") / (1024 ** 3)
print(f"Database size: {db_size:.2f} GB")

Database size: 117.41 GB


1. Listing all tables

In [3]:
conn = sqlite3.connect("stackoverflow.db")

In [4]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables

,name
0,comments
1,posts
2,tags
3,users
4,votes


2. Check columns and schema for each table in the stackoverflow.db

In [5]:
pd.read_sql("PRAGMA table_info(comments)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Id,INTEGER,0,None,1
1,1,PostId,INTEGER,0,None,0
2,2,Score,INTEGER,0,None,0
3,3,Text,TEXT,0,None,0
4,4,CreationDate,TEXT,0,None,0
5,5,UserId,INTEGER,0,None,0


In [6]:
pd.read_sql("PRAGMA table_info(posts)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Id,INTEGER,0,None,1
1,1,PostTypeId,INTEGER,0,None,0
2,2,AcceptedAnswerId,INTEGER,0,None,0
3,3,CreationDate,TEXT,0,None,0
4,4,Score,INTEGER,0,None,0
5,5,ViewCount,INTEGER,0,None,0
6,6,Body,TEXT,0,None,0
7,7,OwnerUserId,INTEGER,0,None,0
8,8,OwnerDisplayName,TEXT,0,None,0
9,9,LastEditorUserId,INTEGER,0,None,0


In [7]:
pd.read_sql("PRAGMA table_info(tags)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Id,INTEGER,0,None,1
1,1,TagName,TEXT,0,None,0
2,2,Count,INTEGER,0,None,0
3,3,ExcerptPostId,INTEGER,0,None,0
4,4,WikiPostId,INTEGER,0,None,0


In [8]:
pd.read_sql("PRAGMA table_info(users)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Id,INTEGER,0,None,1
1,1,Reputation,INTEGER,0,None,0
2,2,CreationDate,TEXT,0,None,0
3,3,DisplayName,TEXT,0,None,0
4,4,LastAccessDate,TEXT,0,None,0
5,5,AboutMe,TEXT,0,None,0
6,6,Views,INTEGER,0,None,0
7,7,UpVotes,INTEGER,0,None,0
8,8,DownVotes,INTEGER,0,None,0


In [9]:
pd.read_sql("PRAGMA table_info(votes)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Id,INTEGER,0,None,1
1,1,PostId,INTEGER,0,None,0
2,2,VoteTypeId,INTEGER,0,None,0
3,3,CreationDate,TEXT,0,None,0


3. Check row count 

In [10]:
tables_list = tables["name"].to_list()
tables_list

['comments', 'posts', 'tags', 'users', 'votes']

In [11]:
for table in tables_list:
    count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table};", conn)
    print(f"{table} has {count['count'][0]:,} rows")

comments has 90,380,323 rows
posts has 59,819,048 rows
tags has 65,675 rows
users has 22,484,235 rows
votes has 195,210,000 rows


4. View first few rows

In [12]:
def preview_table(table, limit=5):
    df = pd.read_sql(f"SELECT *  FROM {table} LIMIT {limit}", conn)
    display(df)

In [13]:
preview_table("comments")

,Id,PostId,Score,Text,CreationDate,UserId
0,10,45651,6,It will help if you give some details of which...,2008-09-06T13:38:23.647,242
1,12,47428,3,One of the things that make a url user-friendl...,2008-09-06T13:51:47.843,4642
2,14,47481,0,"I agree, both CodeRush and RefactorPro are vis...",2008-09-06T14:15:46.897,4642
3,15,47373,0,Just wanted to mention that this is an excelle...,2008-09-06T14:30:40.217,2495
4,16,47497,1,"Indeed, the only way to do this is get the ser...",2008-09-06T14:42:35.303,4642


In [14]:
preview_table("posts")

,Id,PostTypeId,AcceptedAnswerId,CreationDate,Score,ViewCount,Body,OwnerUserId,OwnerDisplayName,LastEditorUserId,LastEditorDisplayName,LastEditDate,LastActivityDate,Title,Tags,AnswerCount,CommentCount,FavoriteCount,ContentLicense
0,4,1,7.0,2008-07-31T21:42:52.667,806,76733.0,<p>I want to assign the decimal variable &quot...,8,None,16124033,Rich B,2022-09-08T05:07:26.033,2022-09-08T05:07:26.033,How to convert Decimal to Double in C#?,|c#|floating-point|type-conversion|double|deci...,13.0,5,0.0,CC BY-SA 4.0
1,6,1,31.0,2008-07-31T22:08:08.620,320,24503.0,<p>I have an absolutely positioned <code>div</...,9,None,9134576,user14723686,2021-01-29T18:46:45.963,2021-01-29T18:46:45.963,Why did the width collapse in the percentage w...,|html|css|internet-explorer-7|,7.0,0,0.0,CC BY-SA 4.0
2,7,2,NaN,2008-07-31T22:17:57.883,529,NaN,<p>An explicit cast to <code>double</code> lik...,9,None,5496973,None,2019-10-21T14:03:54.607,2019-10-21T14:03:54.607,None,None,NaN,0,NaN,CC BY-SA 4.0
3,9,1,1404.0,2008-07-31T23:40:59.743,2248,827598.0,<p>Given a <code>DateTime</code> representing ...,1,None,3524942,Rich B,2022-07-27T22:34:36.320,2024-01-30T07:16:35.683,How do I calculate someone's age based on a Da...,|c#|.net|datetime|,75.0,11,0.0,CC BY-SA 4.0
4,11,1,1248.0,2008-07-31T23:55:37.967,1653,202800.0,<p>Given a specific <code>DateTime</code> valu...,1,None,16790137,user2370523,2022-07-10T00:19:55.237,2023-11-28T13:54:40.813,Calculate relative time in C#,|c#|datetime|time|datediff|relative-time-span|,42.0,4,0.0,CC BY-SA 4.0


In [15]:
preview_table("tags")

,Id,TagName,Count,ExcerptPostId,WikiPostId
0,1,.net,337923,3624959,3607476
1,2,html,1187348,3673183,3673182
2,3,javascript,2528894,3624960,3607052
3,4,css,804268,3644670,3644669
4,5,php,1464496,3624936,3607050


In [16]:
preview_table("users")

,Id,Reputation,CreationDate,DisplayName,LastAccessDate,AboutMe,Views,UpVotes,DownVotes
0,-1017,1,2023-09-15T20:10:32.247,Mobile Development,2023-09-15T20:10:32.290,<p>A collective for developers who want to sha...,0,0,0
1,-1016,1,2023-06-28T20:22:59.967,PHP,2023-06-28T20:22:59.980,<p>A collective where developers working with ...,0,0,0
2,-1015,1,2023-06-28T18:50:43.493,NLP,2023-06-28T18:50:43.527,<p>A collective focused on NLP (natural langua...,0,0,0
3,-1014,1,2023-02-17T19:52:56.213,R Language,2023-02-17T19:52:56.280,<p>A collective where data scientists and AI r...,0,0,0
4,-1013,1,2023-02-17T19:25:19.953,CI/CD,2023-02-17T19:25:20.037,<p>A collective where developers focused on co...,0,0,0


In [17]:
preview_table("votes")

,Id,PostId,VoteTypeId,CreationDate
0,1,1,2,2008-07-31T00:00:00.000
1,2,3,2,2008-07-31T00:00:00.000
2,3,2,2,2008-07-31T00:00:00.000
3,4,4,2,2008-07-31T00:00:00.000
4,5,6,2,2008-07-31T00:00:00.000


5. Null / missing values

In [18]:
def null_counts(table):
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 10000", conn)
    return df.isnull().sum()

In [19]:
null_counts("comments")

Id                0
PostId            0
Score             0
Text              0
CreationDate      0
UserId          214
dtype: int64

In [20]:
null_counts("posts")

Id                          0
PostTypeId                  0
AcceptedAnswerId         8220
CreationDate                0
Score                       0
ViewCount                7914
Body                        0
OwnerUserId               261
OwnerDisplayName         3574
LastEditorUserId         5442
LastEditorDisplayName    7304
LastEditDate             5313
LastActivityDate            0
Title                    7914
Tags                     7914
AnswerCount              7914
CommentCount                0
FavoriteCount            8444
ContentLicense              0
dtype: int64

In [21]:
null_counts("tags")

Id                  0
TagName             0
Count               0
ExcerptPostId    1849
WikiPostId       1849
dtype: int64

In [22]:
null_counts("users")

Id                   0
Reputation           0
CreationDate         0
DisplayName          0
LastAccessDate       0
AboutMe           1140
Views                0
UpVotes              0
DownVotes            0
dtype: int64

In [23]:
null_counts("votes")

Id              0
PostId          0
VoteTypeId      0
CreationDate    0
dtype: int64

6. Data types inference

In [24]:
df = pd.read_sql("SELECT * FROM comments LIMIT 1000", conn)
df.dtypes

Id                int64
PostId            int64
Score             int64
Text             object
CreationDate     object
UserId          float64
dtype: object

In [25]:
df = pd.read_sql("SELECT * FROM posts LIMIT 1000", conn)
df.dtypes

Id                         int64
PostTypeId                 int64
AcceptedAnswerId         float64
CreationDate              object
Score                      int64
ViewCount                float64
Body                      object
OwnerUserId              float64
OwnerDisplayName          object
LastEditorUserId         float64
LastEditorDisplayName     object
LastEditDate              object
LastActivityDate          object
Title                     object
Tags                      object
AnswerCount              float64
CommentCount               int64
FavoriteCount            float64
ContentLicense            object
dtype: object

In [26]:
df = pd.read_sql("SELECT * FROM tags LIMIT 1000", conn)
df.dtypes

Id                 int64
TagName           object
Count              int64
ExcerptPostId    float64
WikiPostId       float64
dtype: object

In [27]:
df = pd.read_sql("SELECT * FROM users LIMIT 1000", conn)
df.dtypes

Id                 int64
Reputation         int64
CreationDate      object
DisplayName       object
LastAccessDate    object
AboutMe           object
Views              int64
UpVotes            int64
DownVotes          int64
dtype: object

In [28]:
df = pd.read_sql("SELECT * FROM votes LIMIT 1000", conn)
df.dtypes

Id               int64
PostId           int64
VoteTypeId       int64
CreationDate    object
dtype: object

7. Check unique values

In [29]:
# pd.read_sql("SELECT DISTINCT PostTypeId FROM posts", conn)

In [ ]:
# pd.read_sql("SELECT DISTINCT VoteTypeId FROM votes", conn)